In [1]:
# ==============================
# Imports
# ==============================
from Bio import SeqIO
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score
)
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
import torch
import random
import numpy as np
from collections import Counter
import os

# ==============================
# 0. Fix seeds
# ==============================
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ==============================
# 1. Load sequences and labels
# ==============================
sequences = [str(record.seq) for record in SeqIO.parse("E:/paper_7_dataset/dataset.txt", "fasta")]
with open("E:/paper_7_dataset/label.txt", "r") as f:
    labels = [int(line.strip()) for line in f]

print(f"Loaded {len(sequences)} sequences and {len(labels)} labels")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    sequences, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"\nAfter train_test_split:\n  Train size: {len(X_train)}\n  Test size: {len(X_test)}")
print("  Train label distribution:", Counter(y_train))
print("  Test  label distribution:", Counter(y_test))

# ==============================
# 2. Load tokenizer and model
# ==============================
local_path = r"C:\huggingface_models\nucleotide_transformer"
tokenizer = AutoTokenizer.from_pretrained(local_path, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(
    local_path,
    num_labels=2,
    trust_remote_code=True
)

# ==============================
# 3. Dataset class
# ==============================
class RNA6MerDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, k=6, max_length=1000):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.k = k
        self.max_length = max_length

    def split_kmers(self, seq):
        seq = seq.upper().replace("U", "T")
        if len(seq) < self.k:
            seq += "N" * (self.k - len(seq))
        kmers = [seq[i:i+self.k] for i in range(0, len(seq), self.k)]
        if len(kmers[-1]) < self.k:
            kmers[-1] += "N" * (self.k - len(kmers[-1]))
        return kmers

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        kmers = self.split_kmers(self.sequences[idx])
        tokens = self.tokenizer(
            kmers[:998],  # 998 + <cls> + <eos> ≈ 1000 tokens
            is_split_into_words=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in tokens.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# ==============================
# 4. Create datasets
# ==============================
train_dataset = RNA6MerDataset(X_train, y_train, tokenizer, k=6, max_length=1000)
test_dataset  = RNA6MerDataset(X_test, y_test, tokenizer, k=6, max_length=1000)
print("\nDatasets created.")
print(f"  Train dataset length: {len(train_dataset)}")
print(f"  Test  dataset length: {len(test_dataset)}")

# Inspect one sequence
sample_seq = X_train[0]
sample_label = y_train[0]
sample_kmers = train_dataset.split_kmers(sample_seq)
print(f"\nSample sequence length: {len(sample_seq)}, label: {sample_label}")
print(f"Number of non-overlapping 6-mers: {len(sample_kmers)}")
print(f"First 20 kmers: {sample_kmers[:20]}")

# ==============================
# 5. Training arguments
# ==============================
training_args = TrainingArguments(
    output_dir="./results",
    seed=42,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=None,
    logging_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    learning_rate=1e-5,
    weight_decay=0.01,
    fp16=True,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
)

# ==============================
# 6. Final checkpoint evaluation
# ==============================
base_model_path = r"C:\huggingface_models\nucleotide_transformer"
final_ckpt = r"./results/checkpoint-9894"   # 👈 set your chosen checkpoint here

# Load model from checkpoint weights
model_ckpt = AutoModelForSequenceClassification.from_pretrained(
    base_model_path,
    num_labels=2,
    trust_remote_code=True,
    state_dict=torch.load(os.path.join(final_ckpt, "pytorch_model.bin"), map_location="cpu")
)

trainer_ckpt = Trainer(
    model=model_ckpt,
    args=training_args,
    eval_dataset=test_dataset,
    tokenizer=tokenizer
)

# Predict
preds_output = trainer_ckpt.predict(test_dataset)
preds = preds_output.predictions.argmax(-1)
labels = preds_output.label_ids
probs = torch.softmax(torch.tensor(preds_output.predictions), dim=-1)[:, 1].numpy()

# Metrics
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
acc = accuracy_score(labels, preds)
auc = roc_auc_score(labels, probs)

tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
sensitivity = recall

# Print results
print(f"\n=== Evaluation for {final_ckpt} ===")
print(f"Accuracy    : {acc:.4f}")
print(f"Precision   : {precision:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"F1-score    : {f1:.4f}")
print(f"AUC         : {auc:.4f}")
print(f"Sensitivity : {sensitivity:.4f}")
print(f"Specificity : {specificity:.4f}")


Loaded 24734 sequences and 24734 labels

After train_test_split:
  Train size: 19787
  Test size: 4947
  Train label distribution: Counter({1: 9894, 0: 9893})
  Test  label distribution: Counter({0: 2474, 1: 2473})


Some weights of the model checkpoint at C:\huggingface_models\nucleotide_transformer were not used when initializing EsmForSequenceClassification: ['lm_head.layer_norm.bias', 'lm_head.dense.weight', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.decoder.weight', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing EsmForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at C:\huggingface_models\nucleotide_transformer and are newly initialized: ['classifier.dense.bias', 'classifier.out_proj.bia


Datasets created.
  Train dataset length: 19787
  Test  dataset length: 4947

Sample sequence length: 2452, label: 1
Number of non-overlapping 6-mers: 409
First 20 kmers: ['AGAGTC', 'CTGGAA', 'GGCTTT', 'GGGGAG', 'CTCCGG', 'GCCGGG', 'CGGATC', 'GCTGCC', 'TGCAGG', 'GAGTCG', 'GGGATG', 'CCAGGT', 'TCCAGC', 'TGAGCA', 'GCGGCC', 'GCCCGC', 'CAGAGT', 'GCCAGT', 'GGCTCC', 'TTGGAG']



=== Evaluation for ./results/checkpoint-9894 ===
Accuracy    : 0.9800
Precision   : 0.9853
Recall      : 0.9745
F1-score    : 0.9799
AUC         : 0.9959
Sensitivity : 0.9745
Specificity : 0.9854


In [14]:
# =====================================================
# 7. Interpretability & Motif Extraction (Captum + IG)
# =====================================================
from captum.attr import IntegratedGradients
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict, Counter
from scipy.stats import fisher_exact
import torch
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_ckpt = model_ckpt.to(device)
model_ckpt.eval()

# --- Use YOUR exact kmerization method ---
def split_kmers(seq, k=6):
    """Use the EXACT same kmerization as your RNA6MerDataset"""
    seq = seq.upper().replace("U", "T")
    if len(seq) < k:
        seq += "N" * (k - len(seq))
    kmers = [seq[i:i+k] for i in range(0, len(seq), k)]
    if len(kmers[-1]) < k:
        kmers[-1] += "N" * (k - len(kmers[-1]))
    return kmers[:998]  # Match your exact truncation

# --- Batch prediction that matches your original evaluation ---
def batch_predict(sequences, tokenizer, model, batch_size=32):
    """Batch prediction function that matches your original preprocessing EXACTLY"""
    model.eval()
    predictions = []
    
    for i in range(0, len(sequences), batch_size):
        batch_seqs = sequences[i:i+batch_size]
        batch_kmers = [split_kmers(seq) for seq in batch_seqs]
        
        # Tokenize EXACTLY like your original code
        tokens = tokenizer(
            batch_kmers,
            is_split_into_words=True,
            max_length=1000,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            outputs = model(**tokens)
            batch_preds = torch.argmax(outputs.logits, dim=-1)
            predictions.extend(batch_preds.cpu().numpy())
    
    return predictions

# --- Integrated Gradients for single sequence ---
def sequence_importance_both(seq, tokenizer, model, k=6, max_length=1000):
    """Calculate importance scores for both classes using Integrated Gradients"""
    # Use YOUR exact kmerization
    kmers = split_kmers(seq, k=k)
    
    # Tokenize EXACTLY like your original code
    tokens = tokenizer(
        kmers,
        is_split_into_words=True,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).to(device)

    input_ids = tokens["input_ids"]
    attention_mask = tokens["attention_mask"]

    # Get embeddings
    embeddings = model.esm.embeddings.word_embeddings(input_ids)
    embeddings.requires_grad_(True)

    scores = {}
    for target_label in [0, 1]:
        def forward_embeds(embeds):
            outputs = model(inputs_embeds=embeds, attention_mask=attention_mask)
            return outputs.logits[:, target_label]

        ig = IntegratedGradients(forward_embeds)
        baseline = torch.zeros_like(embeddings, device=device)
        
        attributions = ig.attribute(
            embeddings,
            baselines=baseline,
            n_steps=20,
            internal_batch_size=2
        )

        attributions = attributions.sum(dim=-1).squeeze(0).cpu().detach().numpy()
        
        # Map kmer scores back to nucleotide positions (non-overlapping)
        nucleotide_scores = np.zeros(len(seq))
        
        for i, score in enumerate(attributions[:len(kmers)]):
            start = i * k  # Non-overlapping - your original method
            end = min(start + k, len(seq))
            # Distribute score evenly across nucleotides in the kmer
            for pos in range(start, end):
                nucleotide_scores[pos] += score / k
        
        scores[target_label] = nucleotide_scores

    return scores, kmers

# --- Extract important motifs ---
def extract_important_motifs(seq, scores, window_size=6, percentile=90):
    """Extract motifs with high importance scores"""
    threshold = np.percentile(np.abs(scores), percentile)
    motifs = {}
    for i in range(len(seq) - window_size + 1):
        window_score = np.mean(np.abs(scores[i:i+window_size]))
        if window_score >= threshold:
            motif = seq[i:i+window_size]
            motifs[motif] = max(motifs.get(motif, 0), window_score)
    return motifs

# --- Motif enrichment analysis ---
# --- Motif enrichment analysis (separate for classes) ---
def motif_enrichment_analysis_separate(pos_motifs_list, neg_motifs_list):
    """Perform Fisher's exact test for motif enrichment - separate analysis for each class"""
    pos_counts = Counter([motif for motifs in pos_motifs_list for motif in motifs.keys()])
    neg_counts = Counter([motif for motifs in neg_motifs_list for motif in motifs.keys()])
    all_motifs = set(pos_counts.keys()).union(neg_counts.keys())

    # Enriched in positive class (more frequent in positive sequences)
    pos_enriched = []
    # Enriched in negative class (more frequent in negative sequences)  
    neg_enriched = []
    
    for motif in all_motifs:
        pos_with = pos_counts.get(motif, 0)
        pos_without = len(pos_motifs_list) - pos_with
        neg_with = neg_counts.get(motif, 0)
        neg_without = len(neg_motifs_list) - neg_with

        # Test for enrichment in positive class
        table_pos = [[pos_with, pos_without], [neg_with, neg_without]]
        odds_ratio_pos, p_value_pos = fisher_exact(table_pos, alternative="greater")
        
        # Test for enrichment in negative class  
        table_neg = [[neg_with, neg_without], [pos_with, pos_without]]
        odds_ratio_neg, p_value_neg = fisher_exact(table_neg, alternative="greater")
        
        if p_value_pos < 0.05:
            pos_enriched.append((motif, pos_with, neg_with, odds_ratio_pos, p_value_pos))
        if p_value_neg < 0.05:
            neg_enriched.append((motif, neg_with, pos_with, odds_ratio_neg, p_value_neg))

    #return (sorted(pos_enriched, key=lambda x: x[4]),  # sort by p-value
            #sorted(neg_enriched, key=lambda x: x[4]))

    return (sorted(pos_enriched, key=lambda x: (-x[3], x[4])),
        sorted(neg_enriched, key=lambda x: (-x[3], x[4])))

# --- Run the interpretability pipeline ---
print("\n=== Running Interpretability Analysis ===")

# Get model predictions using EXACT same method as your original evaluation
model_predictions = batch_predict(X_test, tokenizer, model_ckpt, batch_size=32)

# Compute metrics - should now match your original results exactly
precision = precision_score(y_test, model_predictions)
recall = recall_score(y_test, model_predictions)
accuracy = accuracy_score(y_test, model_predictions)
f1 = f1_score(y_test, model_predictions)

print("\nEvaluation Metrics on FULL test set:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  F1 Score:  {f1:.4f}")

# Analyze a subset of sequences for interpretability
num_analyze = min(50, len(X_test))
print(f"\nAnalyzing {num_analyze} sequences for interpretability...")

pos_motifs = []
neg_motifs = []

for i in range(num_analyze):
    if i % 10 == 0:
        print(f"Processing sequence {i+1}/{num_analyze}")
    
    seq = X_test[i]
    true_label = y_test[i]
    pred_label = model_predictions[i]
    
    try:
        # Get importance scores using YOUR exact preprocessing
        scores, kmers = sequence_importance_both(seq, tokenizer, model_ckpt)
        
        # Extract motifs using the true label's scores
        if true_label == 1:
            important_motifs = extract_important_motifs(seq, scores[1])
            pos_motifs.append(important_motifs)
        else:
            important_motifs = extract_important_motifs(seq, scores[0])
            neg_motifs.append(important_motifs)
        
        # Plot first few examples
        if i < 3:
            plot_importance_scores(seq, scores[0], scores[1], 
                                 f"Seq {i+1} (True: {true_label}, Pred: {pred_label})")
            
    except Exception as e:
        print(f"Error processing sequence {i+1}: {e}")
        continue

# Perform motif enrichment analysis separately for each class
if pos_motifs and neg_motifs:
    print("\n" + "="*60)
    print("MOTIF ENRICHMENT ANALYSIS - SEPARATE BY CLASS")
    print("="*60)
    
    pos_enriched, neg_enriched = motif_enrichment_analysis_separate(pos_motifs, neg_motifs)
    
    # Positive class enriched motifs
    print(f"\n📈 TOP 10 MOTIFS ENRICHED IN POSITIVE CLASS (N={len(pos_motifs)} sequences)")
    print("="*80)
    print("Motif\t\tPos_Count\tNeg_Count\tOdds_Ratio\tP_Value")
    print("-"*80)
    if pos_enriched:
        for motif, pos_count, neg_count, odds_ratio, p_value in pos_enriched[:10]:
            print(f"{motif}\t\t{pos_count}\t\t{neg_count}\t\t{odds_ratio:.4f}\t\t{p_value:.6f}")
    else:
        print("No significantly enriched motifs found for positive class (p < 0.05)")
    
    # Negative class enriched motifs  
    print(f"\n📉 TOP 10 MOTIFS ENRICHED IN NEGATIVE CLASS (N={len(neg_motifs)} sequences)")
    print("="*80)
    print("Motif\t\tNeg_Count\tPos_Count\tOdds_Ratio\tP_Value")
    print("-"*80)
    if neg_enriched:
        for motif, neg_count, pos_count, odds_ratio, p_value in neg_enriched[:10]:
            print(f"{motif}\t\t{neg_count}\t\t{pos_count}\t\t{odds_ratio:.4f}\t\t{p_value:.6f}")
    else:
        print("No significantly enriched motifs found for negative class (p < 0.05)")
    
    # Summary statistics
    print(f"\n📊 SUMMARY STATISTICS")
    print("="*50)
    print(f"Total motifs in positive sequences: {sum(len(m) for m in pos_motifs)}")
    print(f"Total motifs in negative sequences: {sum(len(m) for m in neg_motifs)}")
    print(f"Unique motifs in positive class: {len(set().union(*[set(m.keys()) for m in pos_motifs]))}")
    print(f"Unique motifs in negative class: {len(set().union(*[set(m.keys()) for m in neg_motifs]))}")
    print(f"Significantly enriched in positive class: {len(pos_enriched)} motifs")
    print(f"Significantly enriched in negative class: {len(neg_enriched)} motifs")

# Print final summary
print(f"\n=== Interpretability Summary ===")
print(f"Analyzed {num_analyze} sequences")
print(f"Positive class sequences: {len(pos_motifs)}")
print(f"Negative class sequences: {len(neg_motifs)}")
if pos_motifs or neg_motifs:
    all_motifs = set().union(*[set(m.keys()) for m in pos_motifs + neg_motifs if m])
    print(f"Total unique motifs found: {len(all_motifs)}")
else:
    print("Total unique motifs found: 0")


=== Running Interpretability Analysis ===

Evaluation Metrics on FULL test set:
  Precision: 0.9853
  Recall:    0.9745
  Accuracy:  0.9800
  F1 Score:  0.9799

Analyzing 50 sequences for interpretability...
Processing sequence 1/50
Error processing sequence 1: name 'plot_importance_scores' is not defined
Error processing sequence 2: name 'plot_importance_scores' is not defined
Error processing sequence 3: name 'plot_importance_scores' is not defined
Processing sequence 11/50
Processing sequence 21/50
Processing sequence 31/50
Processing sequence 41/50

MOTIF ENRICHMENT ANALYSIS - SEPARATE BY CLASS

📈 TOP 10 MOTIFS ENRICHED IN POSITIVE CLASS (N=24 sequences)
Motif		Pos_Count	Neg_Count	Odds_Ratio	P_Value
--------------------------------------------------------------------------------
CGATGA		5		0		inf		0.020061
TCATTT		5		0		inf		0.020061
CCAACA		4		0		inf		0.046140
AAAATC		4		0		inf		0.046140
CTTCTC		4		0		inf		0.046140
CGATTT		4		0		inf		0.046140
TTGACA		4		0		inf		0.046140
TCCCAT		4

In [3]:
# --- Motif enrichment analysis (separate for classes) ---
def motif_enrichment_analysis_separate(pos_motifs_list, neg_motifs_list):
    """Perform Fisher's exact test for motif enrichment - separate analysis for each class"""
    pos_counts = Counter([motif for motifs in pos_motifs_list for motif in motifs.keys()])
    neg_counts = Counter([motif for motifs in neg_motifs_list for motif in motifs.keys()])
    all_motifs = set(pos_counts.keys()).union(neg_counts.keys())

    # Enriched in positive class (more frequent in positive sequences)
    pos_enriched = []
    # Enriched in negative class (more frequent in negative sequences)  
    neg_enriched = []
    
    for motif in all_motifs:
        pos_with = pos_counts.get(motif, 0)
        pos_without = len(pos_motifs_list) - pos_with
        neg_with = neg_counts.get(motif, 0)
        neg_without = len(neg_motifs_list) - neg_with

        # Test for enrichment in positive class
        table_pos = [[pos_with, pos_without], [neg_with, neg_without]]
        odds_ratio_pos, p_value_pos = fisher_exact(table_pos, alternative="greater")
        
        # Test for enrichment in negative class  
        table_neg = [[neg_with, neg_without], [pos_with, pos_without]]
        odds_ratio_neg, p_value_neg = fisher_exact(table_neg, alternative="greater")
        
        if p_value_pos < 0.05:
            pos_enriched.append((motif, pos_with, neg_with, odds_ratio_pos, p_value_pos))
        if p_value_neg < 0.05:
            neg_enriched.append((motif, neg_with, pos_with, odds_ratio_neg, p_value_neg))

    return (sorted(pos_enriched, key=lambda x: x[4]),  # sort by p-value
            sorted(neg_enriched, key=lambda x: x[4]))

# --- Save motifs to file function ---
def save_motifs_to_file(motifs_list, filename):
    """Save motifs to a text file, one motif per line"""
    with open(filename, 'w') as f:
        for motif_data in motifs_list:
            motif = motif_data[0]  # Extract just the motif string
            f.write(motif + '\n')
    print(f"Saved {len(motifs_list)} motifs to {filename}")

# --- Run the interpretability pipeline ---
print("\n=== Running Interpretability Analysis ===")

# Get model predictions using EXACT same method as your original evaluation
model_predictions = batch_predict(X_test, tokenizer, model_ckpt, batch_size=32)

# Compute metrics - should now match your original results exactly
precision = precision_score(y_test, model_predictions)
recall = recall_score(y_test, model_predictions)
accuracy = accuracy_score(y_test, model_predictions)
f1 = f1_score(y_test, model_predictions)

print("\nEvaluation Metrics on FULL test set:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  F1 Score:  {f1:.4f}")

# Analyze a subset of sequences for interpretability
num_analyze = min(200, len(X_test))
print(f"\nAnalyzing {num_analyze} sequences for interpretability...")

pos_motifs = []
neg_motifs = []

for i in range(num_analyze):
    if i % 10 == 0:
        print(f"Processing sequence {i+1}/{num_analyze}")
    
    seq = X_test[i]
    true_label = y_test[i]
    pred_label = model_predictions[i]
    
    try:
        # Get importance scores using YOUR exact preprocessing
        scores, kmers = sequence_importance_both(seq, tokenizer, model_ckpt)
        
        # Extract motifs using the true label's scores
        if true_label == 1:
            important_motifs = extract_important_motifs(seq, scores[1])
            pos_motifs.append(important_motifs)
        else:
            important_motifs = extract_important_motifs(seq, scores[0])
            neg_motifs.append(important_motifs)
        
        # Plot first few examples
        if i < 3:
            plot_importance_scores(seq, scores[0], scores[1], 
                                 f"Seq {i+1} (True: {true_label}, Pred: {pred_label})")
            
    except Exception as e:
        print(f"Error processing sequence {i+1}: {e}")
        continue

# Perform motif enrichment analysis separately for each class
if pos_motifs and neg_motifs:
    print("\n" + "="*60)
    print("MOTIF ENRICHMENT ANALYSIS - SEPARATE BY CLASS")
    print("="*60)
    
    pos_enriched, neg_enriched = motif_enrichment_analysis_separate(pos_motifs, neg_motifs)
    
    # Positive class enriched motifs
    print(f"\n📈 TOP 10 MOTIFS ENRICHED IN POSITIVE CLASS (N={len(pos_motifs)} sequences)")
    print("="*80)
    print("Motif\t\tPos_Count\tNeg_Count\tOdds_Ratio\tP_Value")
    print("-"*80)
    if pos_enriched:
        for motif, pos_count, neg_count, odds_ratio, p_value in pos_enriched[:10]:
            print(f"{motif}\t\t{pos_count}\t\t{neg_count}\t\t{odds_ratio:.4f}\t\t{p_value:.6f}")
    else:
        print("No significantly enriched motifs found for positive class (p < 0.05)")
    
    # Negative class enriched motifs  
    print(f"\n📉 TOP 10 MOTIFS ENRICHED IN NEGATIVE CLASS (N={len(neg_motifs)} sequences)")
    print("="*80)
    print("Motif\t\tNeg_Count\tPos_Count\tOdds_Ratio\tP_Value")
    print("-"*80)
    if neg_enriched:
        for motif, neg_count, pos_count, odds_ratio, p_value in neg_enriched[:10]:
            print(f"{motif}\t\t{neg_count}\t\t{pos_count}\t\t{odds_ratio:.4f}\t\t{p_value:.6f}")
    else:
        print("No significantly enriched motifs found for negative class (p < 0.05)")
    
    # Save motifs to files
    print(f"\n💾 SAVING MOTIFS TO FILES")
    print("="*30)
    
    if pos_enriched:
        save_motifs_to_file(pos_enriched, "E:/enriched_positive_motifs.txt")
    else:
        print("No positive motifs to save")
        
    if neg_enriched:
        save_motifs_to_file(neg_enriched, "E:/enriched_negative_motifs.txt")
    else:
        print("No negative motifs to save")
    
    # Summary statistics
    print(f"\n📊 SUMMARY STATISTICS")
    print("="*50)
    print(f"Total motifs in positive sequences: {sum(len(m) for m in pos_motifs)}")
    print(f"Total motifs in negative sequences: {sum(len(m) for m in neg_motifs)}")
    print(f"Unique motifs in positive class: {len(set().union(*[set(m.keys()) for m in pos_motifs]))}")
    print(f"Unique motifs in negative class: {len(set().union(*[set(m.keys()) for m in neg_motifs]))}")
    print(f"Significantly enriched in positive class: {len(pos_enriched)} motifs")
    print(f"Significantly enriched in negative class: {len(neg_enriched)} motifs")

# Print final summary
print(f"\n=== Interpretability Summary ===")
print(f"Analyzed {num_analyze} sequences")
print(f"Positive class sequences: {len(pos_motifs)}")
print(f"Negative class sequences: {len(neg_motifs)}")
if pos_motifs or neg_motifs:
    all_motifs = set().union(*[set(m.keys()) for m in pos_motifs + neg_motifs if m])
    print(f"Total unique motifs found: {len(all_motifs)}")
else:
    print("Total unique motifs found: 0")


=== Running Interpretability Analysis ===

Evaluation Metrics on FULL test set:
  Precision: 0.9853
  Recall:    0.9745
  Accuracy:  0.9800
  F1 Score:  0.9799

Analyzing 200 sequences for interpretability...
Processing sequence 1/200
Error processing sequence 1: name 'plot_importance_scores' is not defined
Error processing sequence 2: name 'plot_importance_scores' is not defined
Error processing sequence 3: name 'plot_importance_scores' is not defined
Processing sequence 11/200
Processing sequence 21/200
Processing sequence 31/200
Processing sequence 41/200
Processing sequence 51/200
Processing sequence 61/200
Processing sequence 71/200
Processing sequence 81/200
Processing sequence 91/200
Processing sequence 101/200
Processing sequence 111/200
Processing sequence 121/200
Processing sequence 131/200
Processing sequence 141/200
Processing sequence 151/200
Processing sequence 161/200
Processing sequence 171/200
Processing sequence 181/200
Processing sequence 191/200

MOTIF ENRICHMENT A

In [10]:
# --- Motif enrichment analysis (separate for classes) ---
def motif_enrichment_analysis_separate(pos_motifs_list, neg_motifs_list):
    """Perform Fisher's exact test for motif enrichment - separate analysis for each class"""
    pos_counts = Counter([motif for motifs in pos_motifs_list for motif in motifs.keys()])
    neg_counts = Counter([motif for motifs in neg_motifs_list for motif in motifs.keys()])
    all_motifs = set(pos_counts.keys()).union(neg_counts.keys())

    # Enriched in positive class (more frequent in positive sequences)
    pos_enriched = []
    # Enriched in negative class (more frequent in negative sequences)  
    neg_enriched = []
    
    for motif in all_motifs:
        pos_with = pos_counts.get(motif, 0)
        pos_without = len(pos_motifs_list) - pos_with
        neg_with = neg_counts.get(motif, 0)
        neg_without = len(neg_motifs_list) - neg_with

        # Test for enrichment in positive class
        table_pos = [[pos_with, pos_without], [neg_with, neg_without]]
        odds_ratio_pos, p_value_pos = fisher_exact(table_pos, alternative="greater")
        
        # Test for enrichment in negative class  
        table_neg = [[neg_with, neg_without], [pos_with, pos_without]]
        odds_ratio_neg, p_value_neg = fisher_exact(table_neg, alternative="greater")
        
        if p_value_pos < 0.05:
            pos_enriched.append((motif, pos_with, neg_with, odds_ratio_pos, p_value_pos))
        if p_value_neg < 0.05:
            neg_enriched.append((motif, neg_with, pos_with, odds_ratio_neg, p_value_neg))

    return (sorted(pos_enriched, key=lambda x: x[4]),  # sort by p-value
            sorted(neg_enriched, key=lambda x: x[4]))

# --- Save motifs to file function ---
def save_motifs_to_file(motifs_list, filename):
    """Save motifs to a text file, one motif per line"""
    with open(filename, 'w') as f:
        for motif_data in motifs_list:
            motif = motif_data[0]  # Extract just the motif string
            f.write(motif + '\n')
    print(f"Saved {len(motifs_list)} motifs to {filename}")

# --- Save sequences to FASTA function ---
def save_sequences_to_fasta(sequences, indices, filename, class_label):
    """Save sequences to a FASTA file"""
    count = 0
    with open(filename, 'w') as f:
        for i, idx in enumerate(indices):
            seq = sequences[idx]
            # Convert back to RNA format (T -> U)
            rna_seq = seq.upper().replace("T", "U")
            f.write(f">seq_{i+1}_idx_{idx}_class_{class_label}\n")
            f.write(f"{rna_seq}\n")
            count += 1
    print(f"Saved {count} class {class_label} sequences to {filename}")
    return count

# --- Run the interpretability pipeline ---
print("\n=== Running Interpretability Analysis ===")

# Get model predictions using EXACT same method as your original evaluation
model_predictions = batch_predict(X_test, tokenizer, model_ckpt, batch_size=32)

# Compute metrics - should now match your original results exactly
precision = precision_score(y_test, model_predictions)
recall = recall_score(y_test, model_predictions)
accuracy = accuracy_score(y_test, model_predictions)
f1 = f1_score(y_test, model_predictions)

print("\nEvaluation Metrics on FULL test set:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  F1 Score:  {f1:.4f}")

# --- Select correctly predicted sequences ---
print(f"\n🔍 SELECTING CORRECTLY PREDICTED SEQUENCES")
print("="*50)

# Identify correctly predicted sequences (true positives and true negatives)
correctly_predicted_indices = []
true_positive_indices = []  # True label=1, Predicted=1
true_negative_indices = []  # True label=0, Predicted=0

for i, (true_label, pred_label) in enumerate(zip(y_test, model_predictions)):
    if true_label == pred_label:  # Correct prediction
        correctly_predicted_indices.append(i)
        if true_label == 1:
            true_positive_indices.append(i)
        else:
            true_negative_indices.append(i)

print(f"Total correctly predicted sequences: {len(correctly_predicted_indices)}")
print(f"True positives (correct class 1): {len(true_positive_indices)}")
print(f"True negatives (correct class 0): {len(true_negative_indices)}")

# Randomly select 100 from each class (if available)
np.random.seed(42)  # For reproducibility

# Select true positives
if len(true_positive_indices) >= 100:
    selected_positive_indices = np.random.choice(true_positive_indices, 100, replace=False)
else:
    selected_positive_indices = true_positive_indices
    print(f"⚠️  Only {len(true_positive_indices)} true positives available, using all")

# Select true negatives  
if len(true_negative_indices) >= 100:
    selected_negative_indices = np.random.choice(true_negative_indices, 100, replace=False)
else:
    selected_negative_indices = true_negative_indices
    print(f"⚠️  Only {len(true_negative_indices)} true negatives available, using all")

print(f"\n✅ SELECTED FOR MOTIF ANALYSIS:")
print(f"  Positive sequences (true positives): {len(selected_positive_indices)}")
print(f"  Negative sequences (true negatives): {len(selected_negative_indices)}")

# Save the selected sequences to FASTA files
print(f"\n💾 SAVING SELECTED SEQUENCES TO FASTA FILES")
print("="*40)

save_sequences_to_fasta(X_test, selected_positive_indices, 
                       "E:/selected_positive_sequences.fasta", class_label=1)
save_sequences_to_fasta(X_test, selected_negative_indices, 
                       "E:/selected_negative_sequences.fasta", class_label=0)

# --- Perform motif analysis on selected sequences ---
print(f"\n🔬 PERFORMING MOTIF ANALYSIS ON SELECTED SEQUENCES")
print("="*50)

pos_motifs = []
neg_motifs = []

# Analyze positive sequences
print(f"Analyzing {len(selected_positive_indices)} positive sequences...")
for i, idx in enumerate(selected_positive_indices):
    if i % 20 == 0:
        print(f"  Processing positive sequence {i+1}/{len(selected_positive_indices)}")
    
    seq = X_test[idx]
    true_label = y_test[idx]
    pred_label = model_predictions[idx]
    
    try:
        # Get importance scores using YOUR exact preprocessing
        scores, kmers = sequence_importance_both(seq, tokenizer, model_ckpt)
        
        # Extract motifs using class 1 scores (since this is a positive sequence)
        important_motifs = extract_important_motifs(seq, scores[1])
        pos_motifs.append(important_motifs)
        
    except Exception as e:
        print(f"Error processing positive sequence {i+1}: {e}")
        continue

# Analyze negative sequences
print(f"Analyzing {len(selected_negative_indices)} negative sequences...")
for i, idx in enumerate(selected_negative_indices):
    if i % 20 == 0:
        print(f"  Processing negative sequence {i+1}/{len(selected_negative_indices)}")
    
    seq = X_test[idx]
    true_label = y_test[idx]
    pred_label = model_predictions[idx]
    
    try:
        # Get importance scores using YOUR exact preprocessing
        scores, kmers = sequence_importance_both(seq, tokenizer, model_ckpt)
        
        # Extract motifs using class 0 scores (since this is a negative sequence)
        important_motifs = extract_important_motifs(seq, scores[0])
        neg_motifs.append(important_motifs)
        
    except Exception as e:
        print(f"Error processing negative sequence {i+1}: {e}")
        continue

# Perform motif enrichment analysis separately for each class
if pos_motifs and neg_motifs:
    print("\n" + "="*60)
    print("MOTIF ENRICHMENT ANALYSIS - SEPARATE BY CLASS")
    print("="*60)
    
    pos_enriched, neg_enriched = motif_enrichment_analysis_separate(pos_motifs, neg_motifs)
    
    # Positive class enriched motifs
    print(f"\n📈 TOP 10 MOTIFS ENRICHED IN POSITIVE CLASS (N={len(selected_positive_indices)} sequences)")
    print("="*80)
    print("Motif\t\tPos_Count\tNeg_Count\tOdds_Ratio\tP_Value")
    print("-"*80)
    if pos_enriched:
        for motif, pos_count, neg_count, odds_ratio, p_value in pos_enriched[:30]:
            print(f"{motif}\t\t{pos_count}\t\t{neg_count}\t\t{odds_ratio:.4f}\t\t{p_value:.6f}")
    else:
        print("No significantly enriched motifs found for positive class (p < 0.05)")
    
    # Negative class enriched motifs  
    print(f"\n📉 TOP 10 MOTIFS ENRICHED IN NEGATIVE CLASS (N={len(selected_negative_indices)} sequences)")
    print("="*80)
    print("Motif\t\tNeg_Count\tPos_Count\tOdds_Ratio\tP_Value")
    print("-"*80)
    if neg_enriched:
        for motif, neg_count, pos_count, odds_ratio, p_value in neg_enriched[:30]:
            print(f"{motif}\t\t{neg_count}\t\t{pos_count}\t\t{odds_ratio:.4f}\t\t{p_value:.6f}")
    else:
        print("No significantly enriched motifs found for negative class (p < 0.05)")
    
    # Save motifs to files
    print(f"\n💾 SAVING MOTIFS TO FILES")
    print("="*30)
    
    if pos_enriched:
        save_motifs_to_file(pos_enriched, "E:/enriched_positive_motifs.txt")
    else:
        print("No positive motifs to save")
        
    if neg_enriched:
        save_motifs_to_file(neg_enriched, "E:/enriched_negative_motifs.txt")
    else:
        print("No negative motifs to save")
    
    # Summary statistics
    print(f"\n📊 SUMMARY STATISTICS")
    print("="*50)
    print(f"Total motifs in positive sequences: {sum(len(m) for m in pos_motifs)}")
    print(f"Total motifs in negative sequences: {sum(len(m) for m in neg_motifs)}")
    print(f"Unique motifs in positive class: {len(set().union(*[set(m.keys()) for m in pos_motifs]))}")
    print(f"Unique motifs in negative class: {len(set().union(*[set(m.keys()) for m in neg_motifs]))}")
    print(f"Significantly enriched in positive class: {len(pos_enriched)} motifs")
    print(f"Significantly enriched in negative class: {len(neg_enriched)} motifs")

# Print final summary
print(f"\n=== Interpretability Summary ===")
print(f"Analyzed {len(selected_positive_indices) + len(selected_negative_indices)} sequences")
print(f"Positive class sequences (true positives): {len(selected_positive_indices)}")
print(f"Negative class sequences (true negatives): {len(selected_negative_indices)}")
print(f"Excluded false positives and false negatives")
if pos_motifs or neg_motifs:
    all_motifs = set().union(*[set(m.keys()) for m in pos_motifs + neg_motifs if m])
    print(f"Total unique motifs found: {len(all_motifs)}")
else:
    print("Total unique motifs found: 0")

print(f"\n💾 FILES CREATED:")
print(f"  E:/selected_positive_sequences.fasta - {len(selected_positive_indices)} true positive sequences")
print(f"  E:/selected_negative_sequences.fasta - {len(selected_negative_indices)} true negative sequences")
print(f"  E:/enriched_positive_motifs.txt - Enriched motifs from true positives")
print(f"  E:/enriched_negative_motifs.txt - Enriched motifs from true negatives")


=== Running Interpretability Analysis ===

Evaluation Metrics on FULL test set:
  Precision: 0.9853
  Recall:    0.9745
  Accuracy:  0.9800
  F1 Score:  0.9799

🔍 SELECTING CORRECTLY PREDICTED SEQUENCES
Total correctly predicted sequences: 4848
True positives (correct class 1): 2410
True negatives (correct class 0): 2438

✅ SELECTED FOR MOTIF ANALYSIS:
  Positive sequences (true positives): 100
  Negative sequences (true negatives): 100

💾 SAVING SELECTED SEQUENCES TO FASTA FILES
Saved 100 class 1 sequences to E:/selected_positive_sequences.fasta
Saved 100 class 0 sequences to E:/selected_negative_sequences.fasta

🔬 PERFORMING MOTIF ANALYSIS ON SELECTED SEQUENCES
Analyzing 100 positive sequences...
  Processing positive sequence 1/100
  Processing positive sequence 21/100
  Processing positive sequence 41/100
  Processing positive sequence 61/100
  Processing positive sequence 81/100
Analyzing 100 negative sequences...
  Processing negative sequence 1/100
  Processing negative sequenc

In [5]:
import RNA
from collections import defaultdict
from scipy.stats import fisher_exact
import numpy as np

# =====================================================
# 8. RNA Structure Analysis for Enriched Motifs
# =====================================================

def analyze_motif_structures(motif_file, sequence_file, output_prefix):
    """Analyze RNA structures for motifs in their native sequences"""
    
    # ---------- Load motifs ----------
    print(f"Loading motifs from {motif_file}...")
    motifs = []
    with open(motif_file, 'r') as f:
        motifs = [line.strip() for line in f if line.strip()]
    
    top_10_motifs = motifs[:30]  # Get top 10 motifs
    print(f"Loaded {len(top_10_motifs)} motifs: {top_10_motifs}")
    
    # ---------- Load sequences ----------
    print(f"Loading sequences from {sequence_file}...")
    sequences = []
    current_seq = ""
    with open(sequence_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if current_seq:
                    sequences.append(current_seq)
                current_seq = ""
            else:
                current_seq += line
        if current_seq:
            sequences.append(current_seq)
    
    print(f"Loaded {len(sequences)} sequences")
    
    # ---------- Predict structures for all sequences ----------
    print("Predicting RNA structures...")
    sequence_structures = []
    for i, seq in enumerate(sequences):
        if i % 20 == 0:
            print(f"  Folding sequence {i+1}/{len(sequences)}")
        struct, mfe = RNA.fold(seq)
        sequence_structures.append(struct)
    
    # ---------- Analyze each motif ----------
    print(f"\nAnalyzing motif structures...")
    motif_analysis_results = {}
    
    for motif in top_10_motifs:
        print(f"  Analyzing motif: {motif}")
        motif_analysis_results[motif] = {
            "total_occurrences": 0,
            "loop_occurrences": 0,
            "stem_occurrences": 0,
            "sequences_with_motif": 0
        }
        
        for seq, struct in zip(sequences, sequence_structures):
            # Find all occurrences of motif in this sequence
            start = 0
            motif_found = False
            
            while start <= len(seq) - len(motif):
                if seq[start:start+len(motif)] == motif:
                    motif_found = True
                    motif_analysis_results[motif]["total_occurrences"] += 1
                    
                    # Analyze structure at motif position
                    motif_struct = struct[start:start+len(motif)]
                    loop_count = motif_struct.count('.')
                    stem_count = len(motif_struct) - loop_count
                    
                    if loop_count > stem_count:
                        motif_analysis_results[motif]["loop_occurrences"] += 1
                    else:
                        motif_analysis_results[motif]["stem_occurrences"] += 1
                    
                    start += len(motif)  # Non-overlapping search
                else:
                    start += 1
            
            if motif_found:
                motif_analysis_results[motif]["sequences_with_motif"] += 1
    
    # ---------- Statistical enrichment analysis ----------
    print(f"\nPerforming statistical enrichment analysis...")
    
    # Calculate background structure distribution
    total_loop_bg = sum(s.count('.') for s in sequence_structures)
    total_stem_bg = sum(len(s) for s in sequence_structures) - total_loop_bg
    
    print(f"Background distribution - Loop: {total_loop_bg}, Stem: {total_stem_bg}")
    
    # Fisher's exact test for each motif
    enrichment_results = []
    
    for motif, counts in motif_analysis_results.items():
        if counts["total_occurrences"] == 0:
            continue
            
        loop_motif = counts["loop_occurrences"]
        stem_motif = counts["stem_occurrences"]
        
        # Calculate non-motif positions
        loop_nonmotif = total_loop_bg - loop_motif
        stem_nonmotif = total_stem_bg - stem_motif
        
        # Fisher's exact test
        table = [[loop_motif, loop_nonmotif],
                 [stem_motif, stem_nonmotif]]
        
        try:
            oddsratio, pvalue = fisher_exact(table, alternative='greater')
            
            # Determine enrichment type
            if loop_motif / (loop_motif + stem_motif) > total_loop_bg / (total_loop_bg + total_stem_bg):
                enrichment_type = "LOOP-ENRICHED"
            else:
                enrichment_type = "STEM-ENRICHED"
                
            enrichment_results.append({
                'motif': motif,
                'loop_count': loop_motif,
                'stem_count': stem_motif,
                'total_occurrences': counts["total_occurrences"],
                'sequences_with_motif': counts["sequences_with_motif"],
                'odds_ratio': oddsratio,
                'p_value': pvalue,
                'enrichment_type': enrichment_type
            })
            
        except ValueError as e:
            print(f"  Error analyzing motif {motif}: {e}")
            continue
    
    # Sort by p-value
    enrichment_results.sort(key=lambda x: x['p_value'])
    
    # ---------- Save results ----------
    output_file = f"{output_prefix}_structure_analysis.txt"
    with open(output_file, 'w') as f:
        f.write("MOTIF STRUCTURE ENRICHMENT ANALYSIS\n")
        f.write("=" * 80 + "\n")
        f.write(f"Motif file: {motif_file}\n")
        f.write(f"Sequence file: {sequence_file}\n")
        f.write(f"Total sequences analyzed: {len(sequences)}\n")
        f.write(f"Total motifs analyzed: {len(top_10_motifs)}\n\n")
        
        f.write("RESULTS:\n")
        f.write("-" * 80 + "\n")
        f.write("Motif\t\tLoop\tStem\tTotal\tSequences\tOdds_Ratio\tP_Value\t\tEnrichment\n")
        f.write("-" * 80 + "\n")
        
        for result in enrichment_results:
            f.write(f"{result['motif']}\t\t"
                   f"{result['loop_count']}\t"
                   f"{result['stem_count']}\t"
                   f"{result['total_occurrences']}\t"
                   f"{result['sequences_with_motif']}\t\t"
                   f"{result['odds_ratio']:.3f}\t\t"
                   f"{result['p_value']:.2e}\t"
                   f"{result['enrichment_type']}\n")
    
    # ---------- Print summary ----------
    print(f"\n📊 STRUCTURE ANALYSIS RESULTS for {output_prefix}")
    print("=" * 80)
    print("Motif\t\tLoop\tStem\tTotal\tSequences\tOdds_Ratio\tP_Value\t\tEnrichment")
    print("-" * 80)
    
    for result in enrichment_results:
        print(f"{result['motif']}\t\t"
              f"{result['loop_count']}\t"
              f"{result['stem_count']}\t"
              f"{result['total_occurrences']}\t"
              f"{result['sequences_with_motif']}\t\t"
              f"{result['odds_ratio']:.3f}\t\t"
              f"{result['p_value']:.2e}\t"
              f"{result['enrichment_type']}")
    
    print(f"\n💾 Results saved to: {output_file}")
    return enrichment_results

# =====================================================
# Run Structure Analysis
# =====================================================

print("\n" + "="*70)
print("RNA STRUCTURE ANALYSIS FOR ENRICHED MOTIFS")
print("="*70)

# Analyze positive motifs in positive sequences
print("\n🔬 ANALYZING POSITIVE MOTIFS IN POSITIVE SEQUENCES")
positive_results = analyze_motif_structures(
    motif_file="E:/enriched_positive_motifs.txt",
    sequence_file="E:/selected_positive_sequences.fasta", 
    output_prefix="positive"
)

# Analyze negative motifs in negative sequences  
print("\n🔬 ANALYZING NEGATIVE MOTIFS IN NEGATIVE SEQUENCES")
negative_results = analyze_motif_structures(
    motif_file="E:/enriched_negative_motifs.txt",
    sequence_file="E:/selected_negative_sequences.fasta",
    output_prefix="negative"
)

# ---------- Comparative Analysis ----------
print("\n" + "="*70)
print("COMPARATIVE ANALYSIS SUMMARY")
print("="*70)

def print_comparative_summary(positive_results, negative_results):
    print("\n📈 POSITIVE MOTIFS (in Positive Sequences):")
    print("-" * 50)
    for result in positive_results[:5]:  # Top 5
        print(f"{result['motif']}: {result['enrichment_type']} "
              f"(p={result['p_value']:.2e}, OR={result['odds_ratio']:.2f})")
    
    print("\n📉 NEGATIVE MOTIFS (in Negative Sequences):")
    print("-" * 50)
    for result in negative_results[:5]:  # Top 5
        print(f"{result['motif']}: {result['enrichment_type']} "
              f"(p={result['p_value']:.2e}, OR={result['odds_ratio']:.2f})")
    
    # Count enrichment types
    pos_loop_enriched = sum(1 for r in positive_results if "LOOP" in r['enrichment_type'])
    pos_stem_enriched = sum(1 for r in positive_results if "STEM" in r['enrichment_type'])
    neg_loop_enriched = sum(1 for r in negative_results if "LOOP" in r['enrichment_type'])
    neg_stem_enriched = sum(1 for r in negative_results if "STEM" in r['enrichment_type'])
    
    print(f"\n📊 ENRICHMENT DISTRIBUTION:")
    print(f"Positive motifs: {pos_loop_enriched} loop-enriched, {pos_stem_enriched} stem-enriched")
    print(f"Negative motifs: {neg_loop_enriched} loop-enriched, {neg_stem_enriched} stem-enriched")

print_comparative_summary(positive_results, negative_results)

print("\n✅ Structure analysis completed!")
print("Files created:")
print("  - positive_structure_analysis.txt")
print("  - negative_structure_analysis.txt")


RNA STRUCTURE ANALYSIS FOR ENRICHED MOTIFS

🔬 ANALYZING POSITIVE MOTIFS IN POSITIVE SEQUENCES
Loading motifs from E:/enriched_positive_motifs.txt...
Loaded 10 motifs: ['AAAAAT', 'CGAAAA', 'CAAATT', 'AAATTT', 'AAACAT', 'GAAAAA', 'CAAAAT', 'TTGATG', 'ATTTAA', 'AAATGT']
Loading sequences from E:/selected_positive_sequences.fasta...
Loaded 100 sequences
Predicting RNA structures...
  Folding sequence 1/100
  Folding sequence 21/100
  Folding sequence 41/100
  Folding sequence 61/100
  Folding sequence 81/100

Analyzing motif structures...
  Analyzing motif: AAAAAT
  Analyzing motif: CGAAAA
  Analyzing motif: CAAATT
  Analyzing motif: AAATTT
  Analyzing motif: AAACAT
  Analyzing motif: GAAAAA
  Analyzing motif: CAAAAT
  Analyzing motif: TTGATG
  Analyzing motif: ATTTAA
  Analyzing motif: AAATGT

Performing statistical enrichment analysis...
Background distribution - Loop: 69325, Stem: 102344

📊 STRUCTURE ANALYSIS RESULTS for positive
Motif		Loop	Stem	Total	Sequences	Odds_Ratio	P_Value		Enr

In [15]:
import RNA
from collections import defaultdict
from scipy.stats import fisher_exact

motif_file = "E:/enriched_positive_motifs.txt"        # one motif per line
pos_seq_file = "E:/selected_positive_sequences.fasta"  # fasta format

motifs = []
sequences = []

# ---------- Load motifs ----------
with open(motif_file) as f:
    motifs = [line.strip() for line in f if line.strip()]

# ---------- Load sequences ----------
with open(pos_seq_file) as f:
    seq = ""
    for line in f:
        line = line.strip()
        if line.startswith(">"):  # fasta header
            if seq:
                sequences.append(seq)
                seq = ""
        else:
            seq += line
    if seq:
        sequences.append(seq)

# ---------- Fold sequences ----------
structures = []
for seq in sequences:
    struct, mfe = RNA.fold(seq)
    structures.append(struct)

# ---------- Map motif occurrences ----------
motif_struct_counts = defaultdict(lambda: {"loop": 0, "stem": 0})

for motif in motifs:
    for seq, struct in zip(sequences, structures):
        start = 0
        while True:
            start = seq.find(motif, start)
            if start == -1:
                break
            substruct = struct[start:start+len(motif)]
            loop_count = substruct.count(".")
            stem_count = len(substruct) - loop_count
            if loop_count > stem_count:
                motif_struct_counts[motif]["loop"] += 1
            else:
                motif_struct_counts[motif]["stem"] += 1
            start += 1

# ---------- Fisher’s exact enrichment ----------
for motif, counts in motif_struct_counts.items():
    loop_motif = counts["loop"]
    stem_motif = counts["stem"]

    loop_total = sum(s.count('.') for s in structures)
    stem_total = sum(len(s) for s in structures) - loop_total

    loop_nonmotif = loop_total - loop_motif
    stem_nonmotif = stem_total - stem_motif

    if loop_total + stem_total == 0:
        continue

    table = [[loop_motif, loop_nonmotif],
             [stem_motif, stem_nonmotif]]

    oddsratio, pvalue = fisher_exact(table)
    print(f"Motif: {motif}, Loop:{loop_motif}, Stem:{stem_motif}, "
          f"Odds ratio:{oddsratio:.2f}, p-value:{pvalue:.3e}")

print(".................")


Motif: CGAAAA, Loop:64, Stem:84, Odds ratio:1.12, p-value:5.029e-01
Motif: GAAAAA, Loop:142, Stem:99, Odds ratio:2.12, p-value:7.575e-09
Motif: GCAAAA, Loop:56, Stem:54, Odds ratio:1.53, p-value:2.557e-02
Motif: AAGACG, Loop:14, Stem:39, Odds ratio:0.53, p-value:4.892e-02
Motif: AAAACC, Loop:56, Stem:29, Odds ratio:2.85, p-value:2.321e-06
Motif: CAACAA, Loop:101, Stem:67, Odds ratio:2.23, p-value:3.394e-07
Motif: AAAAAC, Loop:98, Stem:42, Odds ratio:3.45, p-value:2.159e-12
Motif: CGAAGA, Loop:22, Stem:68, Odds ratio:0.48, p-value:1.754e-03
Motif: ACAAAA, Loop:106, Stem:47, Odds ratio:3.33, p-value:5.913e-13
Motif: ACCGGA, Loop:3, Stem:26, Odds ratio:0.17, p-value:5.233e-04
Motif: AGCGAA, Loop:12, Stem:32, Odds ratio:0.55, p-value:9.059e-02
Motif: ACGAAA, Loop:38, Stem:58, Odds ratio:0.97, p-value:9.174e-01
Motif: CGCCAC, Loop:9, Stem:28, Odds ratio:0.47, p-value:6.374e-02
Motif: CGCGGA, Loop:0, Stem:21, Odds ratio:0.00, p-value:2.179e-05
Motif: CAAAAG, Loop:61, Stem:42, Odds ratio:2.15

In [8]:
#Criteria together

In [13]:
# --- Motif enrichment analysis with dual criteria ---
def motif_enrichment_analysis_dual_criteria(pos_motifs_list, neg_motifs_list, 
                                          p_value_threshold=0.05, odds_ratio_threshold=1.5):
    """Perform Fisher's exact test for motif enrichment with dual significance criteria"""
    pos_counts = Counter([motif for motifs in pos_motifs_list for motif in motifs.keys()])
    neg_counts = Counter([motif for motifs in neg_motifs_list for motif in motifs.keys()])
    all_motifs = set(pos_counts.keys()).union(neg_counts.keys())

    # Enriched in positive class (more frequent in positive sequences)
    pos_enriched_high = []  # High significance (both criteria)
    pos_enriched_stat = []   # Only statistically significant
    pos_enriched_effect = [] # Only effect significant
    
    # Enriched in negative class (more frequent in negative sequences)  
    neg_enriched_high = []   # High significance (both criteria)
    neg_enriched_stat = []    # Only statistically significant
    neg_enriched_effect = []  # Only effect significant
    
    for motif in all_motifs:
        pos_with = pos_counts.get(motif, 0)
        pos_without = len(pos_motifs_list) - pos_with
        neg_with = neg_counts.get(motif, 0)
        neg_without = len(neg_motifs_list) - neg_with

        # Skip if any count is zero (can't calculate proper odds ratio)
        if pos_with == 0 or neg_with == 0 or pos_without == 0 or neg_without == 0:
            continue

        # Test for enrichment in positive class
        table_pos = [[pos_with, pos_without], [neg_with, neg_without]]
        odds_ratio_pos, p_value_pos = fisher_exact(table_pos, alternative="greater")
        
        # Test for enrichment in negative class  
        table_neg = [[neg_with, neg_without], [pos_with, pos_without]]
        odds_ratio_neg, p_value_neg = fisher_exact(table_neg, alternative="greater")
        
        # Positive class enrichment
        stat_sig_pos = p_value_pos < p_value_threshold
        effect_sig_pos = odds_ratio_pos > odds_ratio_threshold
        
        if stat_sig_pos and effect_sig_pos:
            pos_enriched_high.append((motif, pos_with, neg_with, odds_ratio_pos, p_value_pos, "HIGH_SIGNIFICANCE"))
        elif stat_sig_pos:
            pos_enriched_stat.append((motif, pos_with, neg_with, odds_ratio_pos, p_value_pos, "STAT_SIGNIFICANT"))
        elif effect_sig_pos:
            pos_enriched_effect.append((motif, pos_with, neg_with, odds_ratio_pos, p_value_pos, "EFFECT_SIGNIFICANT"))
        
        # Negative class enrichment
        stat_sig_neg = p_value_neg < p_value_threshold
        effect_sig_neg = odds_ratio_neg > odds_ratio_threshold
        
        if stat_sig_neg and effect_sig_neg:
            neg_enriched_high.append((motif, neg_with, pos_with, odds_ratio_neg, p_value_neg, "HIGH_SIGNIFICANCE"))
        elif stat_sig_neg:
            neg_enriched_stat.append((motif, neg_with, pos_with, odds_ratio_neg, p_value_neg, "STAT_SIGNIFICANT"))
        elif effect_sig_neg:
            neg_enriched_effect.append((motif, neg_with, pos_with, odds_ratio_neg, p_value_neg, "EFFECT_SIGNIFICANT"))

    # Combine all categories, with high significance first
    pos_enriched = (sorted(pos_enriched_high, key=lambda x: x[4]) +  # sort by p-value
                   sorted(pos_enriched_stat, key=lambda x: x[4]) +
                   sorted(pos_enriched_effect, key=lambda x: x[3], reverse=True))  # sort by OR
    
    neg_enriched = (sorted(neg_enriched_high, key=lambda x: x[4]) +
                   sorted(neg_enriched_stat, key=lambda x: x[4]) +
                   sorted(neg_enriched_effect, key=lambda x: x[3], reverse=True))

    return pos_enriched, neg_enriched

# --- Save motifs to file function with significance info ---
def save_motifs_to_file_with_significance(motifs_list, filename, significance_type="ALL"):
    """Save motifs to a text file with significance information"""
    if significance_type == "HIGH":
        motifs_to_save = [m for m in motifs_list if m[5] == "HIGH_SIGNIFICANCE"]
    elif significance_type == "STAT":
        motifs_to_save = [m for m in motifs_list if m[5] == "STAT_SIGNIFICANT"]
    elif significance_type == "EFFECT":
        motifs_to_save = [m for m in motifs_list if m[5] == "EFFECT_SIGNIFICANT"]
    else:
        motifs_to_save = motifs_list
    
    with open(filename, 'w') as f:
        for motif_data in motifs_to_save:
            motif = motif_data[0]  # Extract just the motif string
            f.write(motif + '\n')
    print(f"Saved {len(motifs_to_save)} motifs ({significance_type}) to {filename}")
    return len(motifs_to_save)

# --- Save sequences to FASTA function ---
def save_sequences_to_fasta(sequences, indices, filename, class_label):
    """Save sequences to a FASTA file"""
    count = 0
    with open(filename, 'w') as f:
        for i, idx in enumerate(indices):
            seq = sequences[idx]
            # Convert back to RNA format (T -> U)
            rna_seq = seq.upper().replace("T", "U")
            f.write(f">seq_{i+1}_idx_{idx}_class_{class_label}\n")
            f.write(f"{rna_seq}\n")
            count += 1
    print(f"Saved {count} class {class_label} sequences to {filename}")
    return count

# --- Run the interpretability pipeline with dual criteria ---
print("\n=== Running Interpretability Analysis with Dual Criteria ===")
print("Dual Criteria: p-value < 0.05 AND odds ratio > 1.5")
print("="*60)

# Get model predictions using EXACT same method as your original evaluation
model_predictions = batch_predict(X_test, tokenizer, model_ckpt, batch_size=32)

# Compute metrics - should now match your original results exactly
precision = precision_score(y_test, model_predictions)
recall = recall_score(y_test, model_predictions)
accuracy = accuracy_score(y_test, model_predictions)
f1 = f1_score(y_test, model_predictions)

print("\nEvaluation Metrics on FULL test set:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  F1 Score:  {f1:.4f}")

# --- Select correctly predicted sequences ---
print(f"\n🔍 SELECTING CORRECTLY PREDICTED SEQUENCES")
print("="*50)

# Identify correctly predicted sequences (true positives and true negatives)
correctly_predicted_indices = []
true_positive_indices = []  # True label=1, Predicted=1
true_negative_indices = []  # True label=0, Predicted=0

for i, (true_label, pred_label) in enumerate(zip(y_test, model_predictions)):
    if true_label == pred_label:  # Correct prediction
        correctly_predicted_indices.append(i)
        if true_label == 1:
            true_positive_indices.append(i)
        else:
            true_negative_indices.append(i)

print(f"Total correctly predicted sequences: {len(correctly_predicted_indices)}")
print(f"True positives (correct class 1): {len(true_positive_indices)}")
print(f"True negatives (correct class 0): {len(true_negative_indices)}")

# Randomly select 100 from each class (if available)
np.random.seed(42)  # For reproducibility

# Select true positives
if len(true_positive_indices) >= 100:
    selected_positive_indices = np.random.choice(true_positive_indices, 100, replace=False)
else:
    selected_positive_indices = true_positive_indices
    print(f"⚠️  Only {len(true_positive_indices)} true positives available, using all")

# Select true negatives  
if len(true_negative_indices) >= 100:
    selected_negative_indices = np.random.choice(true_negative_indices, 100, replace=False)
else:
    selected_negative_indices = true_negative_indices
    print(f"⚠️  Only {len(true_negative_indices)} true negatives available, using all")

print(f"\n✅ SELECTED FOR MOTIF ANALYSIS:")
print(f"  Positive sequences (true positives): {len(selected_positive_indices)}")
print(f"  Negative sequences (true negatives): {len(selected_negative_indices)}")

# Save the selected sequences to FASTA files
print(f"\n💾 SAVING SELECTED SEQUENCES TO FASTA FILES")
print("="*40)

save_sequences_to_fasta(X_test, selected_positive_indices, 
                       "E:/selected_positive_sequences.fasta", class_label=1)
save_sequences_to_fasta(X_test, selected_negative_indices, 
                       "E:/selected_negative_sequences.fasta", class_label=0)

# --- Perform motif analysis on selected sequences ---
print(f"\n🔬 PERFORMING MOTIF ANALYSIS ON SELECTED SEQUENCES")
print("="*50)

pos_motifs = []
neg_motifs = []

# Analyze positive sequences
print(f"Analyzing {len(selected_positive_indices)} positive sequences...")
for i, idx in enumerate(selected_positive_indices):
    if i % 20 == 0:
        print(f"  Processing positive sequence {i+1}/{len(selected_positive_indices)}")
    
    seq = X_test[idx]
    true_label = y_test[idx]
    pred_label = model_predictions[idx]
    
    try:
        # Get importance scores using YOUR exact preprocessing
        scores, kmers = sequence_importance_both(seq, tokenizer, model_ckpt)
        
        # Extract motifs using class 1 scores (since this is a positive sequence)
        important_motifs = extract_important_motifs(seq, scores[1])
        pos_motifs.append(important_motifs)
        
    except Exception as e:
        print(f"Error processing positive sequence {i+1}: {e}")
        continue

# Analyze negative sequences
print(f"Analyzing {len(selected_negative_indices)} negative sequences...")
for i, idx in enumerate(selected_negative_indices):
    if i % 20 == 0:
        print(f"  Processing negative sequence {i+1}/{len(selected_negative_indices)}")
    
    seq = X_test[idx]
    true_label = y_test[idx]
    pred_label = model_predictions[idx]
    
    try:
        # Get importance scores using YOUR exact preprocessing
        scores, kmers = sequence_importance_both(seq, tokenizer, model_ckpt)
        
        # Extract motifs using class 0 scores (since this is a negative sequence)
        important_motifs = extract_important_motifs(seq, scores[0])
        neg_motifs.append(important_motifs)
        
    except Exception as e:
        print(f"Error processing negative sequence {i+1}: {e}")
        continue

# Perform motif enrichment analysis with dual criteria
if pos_motifs and neg_motifs:
    print("\n" + "="*70)
    print("MOTIF ENRICHMENT ANALYSIS - DUAL CRITERIA (p < 0.05 AND OR > 1.5)")
    print("="*70)
    
    pos_enriched, neg_enriched = motif_enrichment_analysis_dual_criteria(
        pos_motifs, neg_motifs, p_value_threshold=0.05, odds_ratio_threshold=2)
    
    # Count motifs by significance level
    pos_high = len([m for m in pos_enriched if m[5] == "HIGH_SIGNIFICANCE"])
    pos_stat = len([m for m in pos_enriched if m[5] == "STAT_SIGNIFICANT"])
    pos_effect = len([m for m in pos_enriched if m[5] == "EFFECT_SIGNIFICANT"])
    
    neg_high = len([m for m in neg_enriched if m[5] == "HIGH_SIGNIFICANCE"])
    neg_stat = len([m for m in neg_enriched if m[5] == "STAT_SIGNIFICANT"])
    neg_effect = len([m for m in neg_enriched if m[5] == "EFFECT_SIGNIFICANT"])
    
    # Positive class enriched motifs
    print(f"\n📈 MOTIFS ENRICHED IN POSITIVE CLASS (N={len(selected_positive_indices)} sequences)")
    print("="*90)
    print("Significance\tCount\tMotif\t\tPos_Count\tNeg_Count\tOdds_Ratio\tP_Value")
    print("-"*90)
    
    # High significance first
    high_sig_motifs = [m for m in pos_enriched if m[5] == "HIGH_SIGNIFICANCE"][:10]
    for motif, pos_count, neg_count, odds_ratio, p_value, sig_level in high_sig_motifs:
        print(f"HIGH\t\t{len(high_sig_motifs)}\t{motif}\t\t{pos_count}\t\t{neg_count}\t\t{odds_ratio:.4f}\t{p_value:.6f}")
    
    # Statistical significance only
    stat_sig_motifs = [m for m in pos_enriched if m[5] == "STAT_SIGNIFICANT"][:5]
    for motif, pos_count, neg_count, odds_ratio, p_value, sig_level in stat_sig_motifs:
        print(f"STAT ONLY\t{len(stat_sig_motifs)}\t{motif}\t\t{pos_count}\t\t{neg_count}\t\t{odds_ratio:.4f}\t{p_value:.6f}")
    
    # Negative class enriched motifs  
    print(f"\n📉 MOTIFS ENRICHED IN NEGATIVE CLASS (N={len(selected_negative_indices)} sequences)")
    print("="*90)
    print("Significance\tCount\tMotif\t\tNeg_Count\tPos_Count\tOdds_Ratio\tP_Value")
    print("-"*90)
    
    # High significance first
    high_sig_motifs_neg = [m for m in neg_enriched if m[5] == "HIGH_SIGNIFICANCE"][:10]
    for motif, neg_count, pos_count, odds_ratio, p_value, sig_level in high_sig_motifs_neg:
        print(f"HIGH\t\t{len(high_sig_motifs_neg)}\t{motif}\t\t{neg_count}\t\t{pos_count}\t\t{odds_ratio:.4f}\t{p_value:.6f}")
    
    # Statistical significance only
    stat_sig_motifs_neg = [m for m in neg_enriched if m[5] == "STAT_SIGNIFICANT"][:5]
    for motif, neg_count, pos_count, odds_ratio, p_value, sig_level in stat_sig_motifs_neg:
        print(f"STAT ONLY\t{len(stat_sig_motifs_neg)}\t{motif}\t\t{neg_count}\t\t{pos_count}\t\t{odds_ratio:.4f}\t{p_value:.6f}")
    
    # Save motifs to files with different significance levels
    print(f"\n💾 SAVING MOTIFS TO FILES (BY SIGNIFICANCE LEVEL)")
    print("="*50)
    
    # Save high significance motifs
    high_pos_count = save_motifs_to_file_with_significance(pos_enriched, "E:/enriched_positive_motifs_high.txt", "HIGH")
    high_neg_count = save_motifs_to_file_with_significance(neg_enriched, "E:/enriched_negative_motifs_high.txt", "HIGH")
    
    # Save all significant motifs
    all_pos_count = save_motifs_to_file_with_significance(pos_enriched, "E:/enriched_positive_motifs_all.txt", "ALL")
    all_neg_count = save_motifs_to_file_with_significance(neg_enriched, "E:/enriched_negative_motifs_all.txt", "ALL")
    
    # Summary statistics
    print(f"\n📊 DUAL CRITERIA SUMMARY STATISTICS")
    print("="*60)
    print(f"POSITIVE CLASS MOTIFS:")
    print(f"  High significance (p < 0.05 AND OR > 1.5): {pos_high}")
    print(f"  Statistically significant only: {pos_stat}")
    print(f"  Effect significant only: {pos_effect}")
    print(f"  Total significant: {len(pos_enriched)}")
    
    print(f"\nNEGATIVE CLASS MOTIFS:")
    print(f"  High significance (p < 0.05 AND OR > 1.5): {neg_high}")
    print(f"  Statistically significant only: {neg_stat}")
    print(f"  Effect significant only: {neg_effect}")
    print(f"  Total significant: {len(neg_enriched)}")
    
    print(f"\nTotal motifs in positive sequences: {sum(len(m) for m in pos_motifs)}")
    print(f"Total motifs in negative sequences: {sum(len(m) for m in neg_motifs)}")
    print(f"Unique motifs in positive class: {len(set().union(*[set(m.keys()) for m in pos_motifs]))}")
    print(f"Unique motifs in negative class: {len(set().union(*[set(m.keys()) for m in neg_motifs]))}")

# Print final summary
print(f"\n=== Interpretability Summary with Dual Criteria ===")
print(f"Analyzed {len(selected_positive_indices) + len(selected_negative_indices)} sequences")
print(f"Positive class sequences (true positives): {len(selected_positive_indices)}")
print(f"Negative class sequences (true negatives): {len(selected_negative_indices)}")
print(f"Excluded false positives and false negatives")
print(f"Used dual criteria: p < 0.05 AND odds ratio > 1.5")

if pos_motifs or neg_motifs:
    all_motifs = set().union(*[set(m.keys()) for m in pos_motifs + neg_motifs if m])
    print(f"Total unique motifs found: {len(all_motifs)}")
else:
    print("Total unique motifs found: 0")

print(f"\n💾 FILES CREATED:")
print(f"  E:/selected_positive_sequences.fasta - {len(selected_positive_indices)} true positive sequences")
print(f"  E:/selected_negative_sequences.fasta - {len(selected_negative_indices)} true negative sequences")
print(f"  E:/enriched_positive_motifs_high.txt - High significance motifs (p < 0.05 AND OR > 1.5)")
print(f"  E:/enriched_negative_motifs_high.txt - High significance motifs (p < 0.05 AND OR > 1.5)")
print(f"  E:/enriched_positive_motifs_all.txt - All significant motifs")
print(f"  E:/enriched_negative_motifs_all.txt - All significant motifs")


=== Running Interpretability Analysis with Dual Criteria ===
Dual Criteria: p-value < 0.05 AND odds ratio > 1.5

Evaluation Metrics on FULL test set:
  Precision: 0.9853
  Recall:    0.9745
  Accuracy:  0.9800
  F1 Score:  0.9799

🔍 SELECTING CORRECTLY PREDICTED SEQUENCES
Total correctly predicted sequences: 4848
True positives (correct class 1): 2410
True negatives (correct class 0): 2438

✅ SELECTED FOR MOTIF ANALYSIS:
  Positive sequences (true positives): 100
  Negative sequences (true negatives): 100

💾 SAVING SELECTED SEQUENCES TO FASTA FILES
Saved 100 class 1 sequences to E:/selected_positive_sequences.fasta
Saved 100 class 0 sequences to E:/selected_negative_sequences.fasta

🔬 PERFORMING MOTIF ANALYSIS ON SELECTED SEQUENCES
Analyzing 100 positive sequences...
  Processing positive sequence 1/100
  Processing positive sequence 21/100
  Processing positive sequence 41/100
  Processing positive sequence 61/100
  Processing positive sequence 81/100
Analyzing 100 negative sequences

In [12]:
import RNA
from collections import defaultdict
from scipy.stats import fisher_exact
import numpy as np

# =====================================================
# 8. RNA Structure Analysis for Enriched Motifs
# =====================================================

def analyze_motif_structures(motif_file, sequence_file, output_prefix, 
                           p_value_threshold=0.05, odds_ratio_threshold=1.5):
    """Analyze RNA structures for motifs with both p-value and odds ratio criteria"""
    
    # ---------- Load motifs ----------
    print(f"Loading motifs from {motif_file}...")
    motifs = []
    with open(motif_file, 'r') as f:
        motifs = [line.strip() for line in f if line.strip()]
    
    top_10_motifs = motifs[:10]  # Get top 10 motifs
    print(f"Loaded {len(top_10_motifs)} motifs: {top_10_motifs}")
    
    # ---------- Load sequences ----------
    print(f"Loading sequences from {sequence_file}...")
    sequences = []
    current_seq = ""
    with open(sequence_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if current_seq:
                    sequences.append(current_seq)
                current_seq = ""
            else:
                current_seq += line
        if current_seq:
            sequences.append(current_seq)
    
    print(f"Loaded {len(sequences)} sequences")
    
    # ---------- Predict structures for all sequences ----------
    print("Predicting RNA structures...")
    sequence_structures = []
    for i, seq in enumerate(sequences):
        if i % 20 == 0:
            print(f"  Folding sequence {i+1}/{len(sequences)}")
        struct, mfe = RNA.fold(seq)
        sequence_structures.append(struct)
    
    # ---------- Analyze each motif ----------
    print(f"\nAnalyzing motif structures...")
    motif_analysis_results = {}
    
    for motif in top_10_motifs:
        print(f"  Analyzing motif: {motif}")
        motif_analysis_results[motif] = {
            "total_occurrences": 0,
            "loop_occurrences": 0,
            "stem_occurrences": 0,
            "sequences_with_motif": 0
        }
        
        for seq, struct in zip(sequences, sequence_structures):
            # Find all occurrences of motif in this sequence
            start = 0
            motif_found = False
            
            while start <= len(seq) - len(motif):
                if seq[start:start+len(motif)] == motif:
                    motif_found = True
                    motif_analysis_results[motif]["total_occurrences"] += 1
                    
                    # Analyze structure at motif position
                    motif_struct = struct[start:start+len(motif)]
                    loop_count = motif_struct.count('.')
                    stem_count = len(motif_struct) - loop_count
                    
                    if loop_count > stem_count:
                        motif_analysis_results[motif]["loop_occurrences"] += 1
                    else:
                        motif_analysis_results[motif]["stem_occurrences"] += 1
                    
                    start += len(motif)  # Non-overlapping search
                else:
                    start += 1
            
            if motif_found:
                motif_analysis_results[motif]["sequences_with_motif"] += 1
    
    # ---------- Statistical enrichment analysis ----------
    print(f"\nPerforming statistical enrichment analysis...")
    
    # Calculate background structure distribution
    total_loop_bg = sum(s.count('.') for s in sequence_structures)
    total_stem_bg = sum(len(s) for s in sequence_structures) - total_loop_bg
    
    print(f"Background distribution - Loop: {total_loop_bg}, Stem: {total_stem_bg}")
    
    # Fisher's exact test for each motif
    enrichment_results = []
    
    for motif, counts in motif_analysis_results.items():
        if counts["total_occurrences"] == 0:
            continue
            
        loop_motif = counts["loop_occurrences"]
        stem_motif = counts["stem_occurrences"]
        
        # Calculate non-motif positions
        loop_nonmotif = total_loop_bg - loop_motif
        stem_nonmotif = total_stem_bg - stem_motif
        
        # Skip if any cell is zero (can't calculate OR)
        if loop_motif == 0 or stem_motif == 0 or loop_nonmotif == 0 or stem_nonmotif == 0:
            continue
        
        # Fisher's exact test
        table = [[loop_motif, loop_nonmotif],
                 [stem_motif, stem_nonmotif]]
        
        try:
            oddsratio, pvalue = fisher_exact(table, alternative='two-sided')
            
            # Determine enrichment type and significance
            loop_proportion = loop_motif / (loop_motif + stem_motif)
            bg_loop_proportion = total_loop_bg / (total_loop_bg + total_stem_bg)
            
            if loop_proportion > bg_loop_proportion:
                enrichment_type = "LOOP-ENRICHED"
                effect_direction = "positive"
            else:
                enrichment_type = "STEM-ENRICHED" 
                effect_direction = "positive" if oddsratio > 1 else "negative"
            
            # Determine significance level using BOTH criteria
            statistically_significant = pvalue < p_value_threshold
            biologically_significant = oddsratio > odds_ratio_threshold if enrichment_type == "LOOP-ENRICHED" else (1/oddsratio) > odds_ratio_threshold
            
            if statistically_significant and biologically_significant:
                significance_level = "HIGH_SIGNIFICANCE"
            elif statistically_significant:
                significance_level = "STAT_SIGNIFICANT"
            elif biologically_significant:
                significance_level = "EFFECT_SIGNIFICANT"
            else:
                significance_level = "NOT_SIGNIFICANT"
                
            enrichment_results.append({
                'motif': motif,
                'loop_count': loop_motif,
                'stem_count': stem_motif,
                'total_occurrences': counts["total_occurrences"],
                'sequences_with_motif': counts["sequences_with_motif"],
                'odds_ratio': oddsratio,
                'p_value': pvalue,
                'enrichment_type': enrichment_type,
                'significance_level': significance_level,
                'loop_proportion': loop_proportion,
                'bg_loop_proportion': bg_loop_proportion
            })
            
        except ValueError as e:
            print(f"  Error analyzing motif {motif}: {e}")
            continue
    
    # Sort by significance (HIGH_SIGNIFICANT first, then by p-value)
    significance_order = {"HIGH_SIGNIFICANCE": 0, "STAT_SIGNIFICANT": 1, 
                         "EFFECT_SIGNIFICANT": 2, "NOT_SIGNIFICANT": 3}
    enrichment_results.sort(key=lambda x: (significance_order[x['significance_level']], x['p_value']))
    
    # ---------- Save results ----------
    output_file = f"{output_prefix}_structure_analysis.txt"
    with open(output_file, 'w') as f:
        f.write("MOTIF STRUCTURE ENRICHMENT ANALYSIS\n")
        f.write("=" * 100 + "\n")
        f.write(f"Motif file: {motif_file}\n")
        f.write(f"Sequence file: {sequence_file}\n")
        f.write(f"Total sequences analyzed: {len(sequences)}\n")
        f.write(f"Total motifs analyzed: {len(top_10_motifs)}\n")
        f.write(f"Significance thresholds: p < {p_value_threshold}, OR > {odds_ratio_threshold}\n\n")
        
        f.write("RESULTS (Sorted by Significance):\n")
        f.write("-" * 100 + "\n")
        f.write("Motif\t\tLoop\tStem\tTotal\tOR\t\tP_Value\t\tSignificance\t\tType\n")
        f.write("-" * 100 + "\n")
        
        for result in enrichment_results:
            f.write(f"{result['motif']}\t\t"
                   f"{result['loop_count']}\t"
                   f"{result['stem_count']}\t"
                   f"{result['total_occurrences']}\t"
                   f"{result['odds_ratio']:.2f}\t\t"
                   f"{result['p_value']:.2e}\t"
                   f"{result['significance_level']}\t"
                   f"{result['enrichment_type']}\n")
        
        # Summary statistics
        f.write(f"\nSUMMARY:\n")
        f.write(f"High significance (p < {p_value_threshold} AND OR > {odds_ratio_threshold}): ")
        f.write(f"{sum(1 for r in enrichment_results if r['significance_level'] == 'HIGH_SIGNIFICANCE')}\n")
        f.write(f"Statistically significant only: ")
        f.write(f"{sum(1 for r in enrichment_results if r['significance_level'] == 'STAT_SIGNIFICANT')}\n")
        f.write(f"Effect significant only: ")
        f.write(f"{sum(1 for r in enrichment_results if r['significance_level'] == 'EFFECT_SIGNIFICANT')}\n")
        f.write(f"Not significant: ")
        f.write(f"{sum(1 for r in enrichment_results if r['significance_level'] == 'NOT_SIGNIFICANT')}\n")
    
    # ---------- Print summary ----------
    print(f"\n📊 STRUCTURE ANALYSIS RESULTS for {output_prefix}")
    print("=" * 100)
    print("Motif\t\tLoop\tStem\tTotal\tOR\tP_Value\t\tSignificance\t\tType")
    print("-" * 100)
    
    for result in enrichment_results:
        print(f"{result['motif']}\t\t"
              f"{result['loop_count']}\t"
              f"{result['stem_count']}\t"
              f"{result['total_occurrences']}\t"
              f"{result['odds_ratio']:.2f}\t"
              f"{result['p_value']:.2e}\t"
              f"{result['significance_level']}\t"
              f"{result['enrichment_type']}")
    
    print(f"\n💾 Results saved to: {output_file}")
    return enrichment_results

# =====================================================
# Run Structure Analysis with Dual Criteria
# =====================================================

print("\n" + "="*80)
print("RNA STRUCTURE ANALYSIS WITH DUAL SIGNIFICANCE CRITERIA")
print("="*80)
print("Criteria: p < 0.05 AND odds ratio > 1.5")
print("="*80)

# Analyze with dual criteria
positive_results = analyze_motif_structures(
    motif_file="E:/enriched_positive_motifs.txt",
    sequence_file="E:/selected_positive_sequences.fasta", 
    output_prefix="positive",
    p_value_threshold=0.05,
    odds_ratio_threshold=1.5
)

negative_results = analyze_motif_structures(
    motif_file="E:/enriched_negative_motifs.txt",
    sequence_file="E:/selected_negative_sequences.fasta",
    output_prefix="negative",
    p_value_threshold=0.05, 
    odds_ratio_threshold=1.5
)

# ---------- High Confidence Results ----------
print("\n" + "="*80)
print("HIGH CONFIDENCE RESULTS (p < 0.05 AND OR > 1.5)")
print("="*80)

high_conf_positive = [r for r in positive_results if r['significance_level'] == 'HIGH_SIGNIFICANCE']
high_conf_negative = [r for r in negative_results if r['significance_level'] == 'HIGH_SIGNIFICANCE']

print(f"\n🔬 HIGH CONFIDENCE POSITIVE MOTIFS: {len(high_conf_positive)}")
for result in high_conf_positive:
    print(f"  {result['motif']}: {result['enrichment_type']} "
          f"(OR={result['odds_ratio']:.2f}, p={result['p_value']:.2e})")

print(f"\n🔬 HIGH CONFIDENCE NEGATIVE MOTIFS: {len(high_conf_negative)}")
for result in high_conf_negative:
    print(f"  {result['motif']}: {result['enrichment_type']} "
          f"(OR={result['odds_ratio']:.2f}, p={result['p_value']:.2e})")

print(f"\n✅ Analysis completed with dual significance criteria!")


RNA STRUCTURE ANALYSIS WITH DUAL SIGNIFICANCE CRITERIA
Criteria: p < 0.05 AND odds ratio > 1.5
Loading motifs from E:/enriched_positive_motifs.txt...
Loaded 10 motifs: ['AAAAAT', 'CGAAAA', 'CAAATT', 'AAATTT', 'AAACAT', 'GAAAAA', 'CAAAAT', 'TTGATG', 'ATTTAA', 'AAATGT']
Loading sequences from E:/selected_positive_sequences.fasta...
Loaded 100 sequences
Predicting RNA structures...
  Folding sequence 1/100
  Folding sequence 21/100
  Folding sequence 41/100
  Folding sequence 61/100
  Folding sequence 81/100

Analyzing motif structures...
  Analyzing motif: AAAAAT
  Analyzing motif: CGAAAA
  Analyzing motif: CAAATT
  Analyzing motif: AAATTT
  Analyzing motif: AAACAT
  Analyzing motif: GAAAAA
  Analyzing motif: CAAAAT
  Analyzing motif: TTGATG
  Analyzing motif: ATTTAA
  Analyzing motif: AAATGT

Performing statistical enrichment analysis...
Background distribution - Loop: 69325, Stem: 102344

📊 STRUCTURE ANALYSIS RESULTS for positive
Motif		Loop	Stem	Total	OR	P_Value		Significance		Type
-


KeyboardInterrupt

